<a href="https://colab.research.google.com/github/Anushadhirde/Urban-Heat-Island-Change-Detection/blob/main/Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


This cell mounts your Google Drive to the Colab environment, allowing the notebook to access files stored in your Drive. This is a common first step when working with datasets located in Google Drive.

In [ ]:
!pip install rasterio pandas -q

This cell installs the necessary Python libraries: `rasterio` for working with raster data and `pandas` for data manipulation, particularly for creating and managing dataframes.

In [ ]:
import os

DATA_FOLDER = "/content/drive/MyDrive/DATASET"

tif_files = []

for root, dirs, files in os.walk(DATA_FOLDER):
    for file in files:
        if file.lower().endswith((".tif", ".tiff")):
            tif_files.append(os.path.join(root, file))

print("Total TIFF files found:", len(tif_files))

print("\nFiles by folder:")
for folder in sorted(set(os.path.dirname(f) for f in tif_files)):
    count = sum(os.path.dirname(f) == folder for f in tif_files)
    print(f"{folder}: {count}")

Total TIFF files found: 102

Files by folder:
/content/drive/MyDrive/DATASET: 3
/content/drive/MyDrive/DATASET/Albedo/summer: 9
/content/drive/MyDrive/DATASET/Albedo/winter: 9
/content/drive/MyDrive/DATASET/LST/summer: 9
/content/drive/MyDrive/DATASET/LST/winter: 9
/content/drive/MyDrive/DATASET/LULC: 9
/content/drive/MyDrive/DATASET/NDBI/summer: 9
/content/drive/MyDrive/DATASET/NDBI/winter: 9
/content/drive/MyDrive/DATASET/NDVI/summer: 9
/content/drive/MyDrive/DATASET/NDVI/winter: 9
/content/drive/MyDrive/DATASET/NDWI/summer: 9
/content/drive/MyDrive/DATASET/NDWI/winter: 9


This code block sets up the `DATA_FOLDER` variable, which points to the location of your raw TIFF files on Google Drive. It then walks through this folder and its subdirectories to find all `.tif` and `.tiff` files, printing a count of the total files found and how many files are in each subfolder. This gives you an initial overview of your dataset structure.

In [ ]:
import os
import pandas as pd
import rasterio

DATA_FOLDER = "/content/drive/MyDrive/DATASET"

records = []

for root, dirs, files in os.walk(DATA_FOLDER):
    for file in files:

        if not file.lower().endswith((".tif", ".tiff")):
            continue

        path = os.path.join(root, file)

        try:
            with rasterio.open(path) as src:

                # Calculate valid percentage
                data = src.read(1, masked=True)

                total_pixels = src.width * src.height
                valid_pixels = data.count()

                valid_percent = (
                    valid_pixels / total_pixels * 100
                    if total_pixels > 0 else 0
                )

                records.append({
                    "File": file,
                    "Folder": os.path.relpath(root, DATA_FOLDER),
                    "CRS": str(src.crs),
                    "Width": src.width,
                    "Height": src.height,
                    "Pixel_X": src.transform.a,
                    "Pixel_Y": abs(src.transform.e),
                    "Dtype": src.dtypes[0],
                    "NoData": src.nodata,
                    "Min": float(data.min()) if valid_pixels > 0 else None,
                    "Max": float(data.max()) if valid_pixels > 0 else None,
                    "Valid_%": round(valid_percent, 2)
                })

        except Exception as e:

            records.append({
                "File": file,
                "Folder": os.path.relpath(root, DATA_FOLDER),
                "CRS": "ERROR",
                "Width": None,
                "Height": None,
                "Pixel_X": None,
                "Pixel_Y": None,
                "Dtype": None,
                "NoData": None,
                "Min": None,
                "Max": None,
                "Valid_%": None
            })

df = pd.DataFrame(records)

print("Total files audited:", len(df))

display(df)

Total files audited: 102


,File,Folder,CRS,Width,Height,Pixel_X,Pixel_Y,Dtype,NoData,Min,Max,Valid_%
0,Elevation_Nagpur.tif,.,EPSG:4326,5232,4238,0.000269,0.000269,int16,None,0.0,609.0,100.0
1,Slope_Nagpur.tif,.,EPSG:4326,5232,4238,0.000269,0.000269,float32,None,NaN,NaN,100.0
2,Aspect_Nagpur.tif,.,EPSG:4326,5232,4238,0.000269,0.000269,float32,None,NaN,NaN,100.0
3,LST_2018_Summer_Nagpur.tif,LST/summer,EPSG:4326,5231,4239,0.000269,0.000269,float64,None,NaN,NaN,100.0
4,LST_2019_Summer_Nagpur.tif,LST/summer,EPSG:4326,5231,4239,0.000269,0.000269,float64,None,NaN,NaN,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...
97,LULC_2021_Nagpur.tif,LULC,EPSG:4326,5231,4239,0.000269,0.000269,uint8,None,0.0,8.0,100.0
98,LULC_2025_Nagpur.tif,LULC,EPSG:4326,5231,4239,0.000269,0.000269,uint8,None,0.0,8.0,100.0
99,LULC_2022_Nagpur.tif,LULC,EPSG:4326,5231,4239,0.000269,0.000269,uint8,None,0.0,8.0,100.0
100,LULC_2024_Nagpur.tif,LULC,EPSG:4326,5231,4239,0.000269,0.000269,uint8,None,0.0,8.0,100.0


This cell performs an initial audit of the raw TIFF files. It iterates through each `.tif` file, opens it with `rasterio`, and extracts various metadata such as CRS, dimensions (width, height), pixel size, data type, NoData value, and the percentage of valid pixels. This information is then compiled into a pandas DataFrame, providing a structured overview of the raw dataset's characteristics. Any files that cannot be opened are marked with an 'ERROR' status.

In [ ]:
print("CRS:")
print(df["CRS"].value_counts())

print("\nPixel sizes:")
print(df[["Pixel_X", "Pixel_Y"]].drop_duplicates())

print("\nData types:")
print(df["Dtype"].value_counts())

print("\nNoData values:")
print(df["NoData"].value_counts(dropna=False))

print("\nDimensions:")
print(df[["Width", "Height"]].drop_duplicates())

CRS:
CRS
EPSG:4326    102
Name: count, dtype: int64

Pixel sizes:
    Pixel_X   Pixel_Y
0  0.000269  0.000269
3  0.000269  0.000269

Data types:
Dtype
float32    65
float64    27
uint8       9
int16       1
Name: count, dtype: int64

NoData values:
NoData
None    102
Name: count, dtype: int64

Dimensions:
   Width  Height
0   5232    4238
3   5231    4239


Building on the audit performed in the previous cell, this cell prints summaries of key metadata from the `df` DataFrame. It shows the unique Coordinate Reference Systems (CRS), pixel sizes, data types, NoData values, and dimensions found across all raw TIFF files. This helps to quickly identify any inconsistencies or variations in the raw dataset that might need to be addressed during preprocessing.

In [ ]:
import os
import rasterio
import pandas as pd

DATA_FOLDER = "/content/drive/MyDrive/DATASET"

rows = []

for root, dirs, files in os.walk(DATA_FOLDER):

    for file in files:

        if not file.lower().endswith((".tif", ".tiff")):
            continue

        path = os.path.join(root, file)

        with rasterio.open(path) as src:

            bounds = src.bounds

            rows.append({
                "File": file,
                "Folder": os.path.relpath(root, DATA_FOLDER),
                "Width": src.width,
                "Height": src.height,
                "Pixel_X": src.transform.a,
                "Pixel_Y": abs(src.transform.e),
                "Left": bounds.left,
                "Bottom": bounds.bottom,
                "Right": bounds.right,
                "Top": bounds.top
            })

grid_df = pd.DataFrame(rows)

display(grid_df)

,File,Folder,Width,Height,Pixel_X,Pixel_Y,Left,Bottom,Right,Top
0,Elevation_Nagpur.tif,.,5232,4238,0.000269,0.000269,78.251132,20.580627,79.661128,21.722745
1,Slope_Nagpur.tif,.,5232,4238,0.000269,0.000269,78.251132,20.580627,79.661128,21.722745
2,Aspect_Nagpur.tif,.,5232,4238,0.000269,0.000269,78.251132,20.580627,79.661128,21.722745
3,LST_2018_Summer_Nagpur.tif,LST/summer,5231,4239,0.000269,0.000269,78.251256,20.580493,79.660982,21.722881
4,LST_2019_Summer_Nagpur.tif,LST/summer,5231,4239,0.000269,0.000269,78.251256,20.580493,79.660982,21.722881
...,...,...,...,...,...,...,...,...,...,...
97,LULC_2021_Nagpur.tif,LULC,5231,4239,0.000269,0.000269,78.251256,20.580493,79.660982,21.722881
98,LULC_2025_Nagpur.tif,LULC,5231,4239,0.000269,0.000269,78.251256,20.580493,79.660982,21.722881
99,LULC_2022_Nagpur.tif,LULC,5231,4239,0.000269,0.000269,78.251256,20.580493,79.660982,21.722881
100,LULC_2024_Nagpur.tif,LULC,5231,4239,0.000269,0.000269,78.251256,20.580493,79.660982,21.722881


This cell focuses on extracting spatial boundary information from each raw TIFF file. It opens each file, retrieves its `bounds` (left, bottom, right, top coordinates), and stores this along with other metadata (file name, folder, width, height, pixel size) in a pandas DataFrame called `grid_df`. This DataFrame is crucial for understanding the geographical extent and spatial relationships of your raw data.

In [ ]:
print("Unique widths:")
print(grid_df["Width"].unique())

print("\nUnique heights:")
print(grid_df["Height"].unique())

print("\nX resolution:")
print(grid_df["Pixel_X"].unique())

print("\nY resolution:")
print(grid_df["Pixel_Y"].unique())

print("\nOverall extent:")
print("Left  :", grid_df["Left"].min())
print("Bottom:", grid_df["Bottom"].min())
print("Right :", grid_df["Right"].max())
print("Top   :", grid_df["Top"].max())

Unique widths:
[5232 5231]

Unique heights:
[4238 4239]

X resolution:
[0.00026949 0.00026949]

Y resolution:
[0.00026949 0.00026949]

Overall extent:
Left  : 78.25113225092758
Bottom: 20.580492990706652
Right : 79.66112792088158
Top   : 21.722880537521448


This cell analyzes the `grid_df` created in the previous step to identify unique characteristics of the raster grids. It prints all unique widths, heights, X resolutions, and Y resolutions. Most importantly, it calculates and prints the overall extent (minimum left, minimum bottom, maximum right, maximum top) across all TIFF files. This gives you a clear picture of the full geographical coverage of your dataset.

In [ ]:
import rasterio

MASTER = "/content/drive/MyDrive/DATASET/LST/summer/LST_2017_Summer_Nagpur.tif"

with rasterio.open(MASTER) as src:
    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)


CRS: EPSG:4326
Width: 5231
Height: 4239
Resolution: (0.00026949458523585647, 0.00026949458523585647)
Bounds: BoundingBox(left=78.251256252839, bottom=20.580492990706652, right=79.66098242820776, top=21.722880537521448)


This cell selects a specific TIFF file (`MASTER`) to serve as a reference for the desired output grid. It opens this master file and prints its CRS, width, height, resolution, and bounds. This information will be used to define the target CRS, resolution, and grid for all other rasters during the reprojection process.

In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform
from rasterio.crs import CRS

MASTER = "/content/drive/MyDrive/DATASET/LST/summer/LST_2017_Summer_Nagpur.tif"

TARGET_CRS = "EPSG:32644"
TARGET_RES = 30  # metres

with rasterio.open(MASTER) as src:

    transform, width, height = calculate_default_transform(
        src.crs,
        TARGET_CRS,
        src.width,
        src.height,
        *src.bounds,
        resolution=TARGET_RES
    )

    print("TARGET CRS:", TARGET_CRS)
    print("TARGET RESOLUTION:", TARGET_RES, "m")
    print("TARGET WIDTH:", width)
    print("TARGET HEIGHT:", height)
    print("TARGET TRANSFORM:")
    print(transform)

    print("\nTARGET EXTENT:")
    print(
        transform.c,
        transform.f - height * abs(transform.e),
        transform.c + width * transform.a,
        transform.f
    )

TARGET CRS: EPSG:32644
TARGET RESOLUTION: 30 m
TARGET WIDTH: 4936
TARGET HEIGHT: 4280
TARGET TRANSFORM:
| 30.00, 0.00, 213452.43|
| 0.00,-30.00, 2404680.12|
| 0.00, 0.00, 1.00|

TARGET EXTENT:
213452.42500368197 2276280.1160590868 361532.425003682 2404680.1160590868


Using the `MASTER` file as a reference, this cell calculates the parameters for the target output grid. It defines a `TARGET_CRS` (EPSG:32644) and a `TARGET_RES` (30 meters). The `rasterio.warp.calculate_default_transform` function is then used to determine the necessary transformation matrix, width, and height required to reproject the master file to the target CRS and resolution. This cell outputs these calculated target grid parameters, which are essential for standardizing all other raster files.

In [ ]:
import os
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np

# ============================================================
# PATHS
# ============================================================

DATA_FOLDER = "/content/drive/MyDrive/DATASET"
OUTPUT_FOLDER = "/content/drive/MyDrive/DATASET/PROCESSED_30M"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


# ============================================================
# MASTER GRID
# ============================================================

MASTER = "/content/drive/MyDrive/DATASET/LST/summer/LST_2017_Summer_Nagpur.tif"

TARGET_CRS = "EPSG:32644"
TARGET_RES = 30

with rasterio.open(MASTER) as src:

    from rasterio.warp import calculate_default_transform

    transform, width, height = calculate_default_transform(
        src.crs,
        TARGET_CRS,
        src.width,
        src.height,
        *src.bounds,
        resolution=TARGET_RES
    )

print("TARGET GRID")
print("---------------------------")
print("CRS       :", TARGET_CRS)
print("Resolution:", TARGET_RES, "m")
print("Width     :", width)
print("Height    :", height)
print("Transform :", transform)
print()


# ============================================================
# FIND ALL TIFF FILES
# ============================================================

tif_files = []

for root, dirs, files in os.walk(DATA_FOLDER):

    # Don't process files that are already in output folder
    if os.path.abspath(root).startswith(os.path.abspath(OUTPUT_FOLDER)):
        continue

    for file in files:

        if file.lower().endswith((".tif", ".tiff")):

            tif_files.append(
                os.path.join(root, file)
            )

print("Total TIFF files found:", len(tif_files))
print()


# ============================================================
# RESAMPLING METHOD
# ============================================================

def get_resampling_method(path):

    path_lower = path.lower()

    # Categorical data
    if "lulc" in path_lower:
        return Resampling.nearest

    # Continuous data
    return Resampling.bilinear


# ============================================================
# PROCESS ONE FILE
# ============================================================

def process_raster(input_path):

    try:

        # Preserve folder structure
        relative_path = os.path.relpath(
            input_path,
            DATA_FOLDER
        )

        output_path = os.path.join(
            OUTPUT_FOLDER,
            relative_path
        )

        output_dir = os.path.dirname(output_path)

        os.makedirs(output_dir, exist_ok=True)

        # Add "_30m" to filename
        filename = os.path.basename(output_path)

        name, ext = os.path.splitext(filename)

        output_path = os.path.join(
            output_dir,
            name + "_30m.tif"
        )

        # Skip if already processed
        if os.path.exists(output_path):

            return (
                input_path,
                "SKIPPED",
                output_path
            )

        resampling_method = get_resampling_method(input_path)

        with rasterio.open(input_path) as src:

            # ------------------------------------------------
            # Choose output data type
            # ------------------------------------------------

            if "lulc" in input_path.lower():

                output_dtype = "uint8"
                output_nodata = 255

            else:

                output_dtype = "float32"
                output_nodata = np.nan


            # ------------------------------------------------
            # Output metadata
            # ------------------------------------------------

            profile = src.profile.copy()

            profile.update({

                "driver": "GTiff",

                "crs": TARGET_CRS,

                "transform": transform,

                "width": width,

                "height": height,

                "count": src.count,

                "dtype": output_dtype,

                "nodata": output_nodata,

                "compress": "LZW",

                "BIGTIFF": "IF_SAFER",

                "tiled": True,

                "blockxsize": 256,

                "blockysize": 256

            })


            # ------------------------------------------------
            # Reproject
            # ------------------------------------------------

            with rasterio.open(
                output_path,
                "w",
                **profile
            ) as dst:

                for band in range(1, src.count + 1):

                    destination = np.empty(
                        (height, width),
                        dtype=output_dtype
                    )

                    # Initialize NoData
                    if np.issubdtype(
                        np.dtype(output_dtype),
                        np.floating
                    ):
                        destination.fill(np.nan)

                    else:
                        destination.fill(output_nodata)


                    reproject(

                        source=rasterio.band(
                            src,
                            band
                        ),

                        destination=destination,

                        src_transform=src.transform,

                        src_crs=src.crs,

                        src_nodata=src.nodata,

                        dst_transform=transform,

                        dst_crs=TARGET_CRS,

                        dst_nodata=output_nodata,

                        resampling=resampling_method,

                        num_threads=2

                    )

                    dst.write(
                        destination,
                        band
                    )

        return (
            input_path,
            "DONE",
            output_path
        )

    except Exception as e:

        return (
            input_path,
            "ERROR: " + str(e),
            None
        )


# ============================================================
# PROCESS ALL FILES
# ============================================================

print("Starting processing...")
print("This may take some time because files are being read")
print("and written directly through Google Drive.")
print()

# Use several CPU workers
MAX_WORKERS = 4

completed = 0
errors = 0

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    futures = {
        executor.submit(
            process_raster,
            path
        ): path
        for path in tif_files
    }

    for future in as_completed(futures):

        input_path, status, output_path = future.result()

        completed += 1

        if status.startswith("ERROR"):
            errors += 1

        print(
            f"[{completed}/{len(tif_files)}] "
            f"{status} - "
            f"{os.path.basename(input_path)}"
        )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 60)
print("PROCESSING COMPLETE")
print("=" * 60)

print("Input files :", len(tif_files))
print("Completed   :", completed)
print("Errors      :", errors)
print()
print("Output folder:")
print(OUTPUT_FOLDER)

TARGET GRID
---------------------------
CRS       : EPSG:32644
Resolution: 30 m
Width     : 4936
Height    : 4280
Transform : | 30.00, 0.00, 213452.43|
| 0.00,-30.00, 2404680.12|
| 0.00, 0.00, 1.00|

Total TIFF files found: 102

Starting processing...
This may take some time because files are being read
and written directly through Google Drive.

[1/102] DONE - Elevation_Nagpur.tif
[2/102] DONE - Slope_Nagpur.tif
[3/102] DONE - Aspect_Nagpur.tif
[4/102] DONE - LST_2018_Summer_Nagpur.tif
[5/102] DONE - LST_2019_Summer_Nagpur.tif
[6/102] DONE - LST_2022_Summer_Nagpur.tif
[7/102] DONE - LST_2017_Summer_Nagpur.tif
[8/102] DONE - LST_2020_Summer_Nagpur.tif
[9/102] DONE - LST_2024_Summer_Nagpur.tif
[10/102] DONE - LST_2023_Summer_Nagpur.tif
[11/102] DONE - LST_2021_Summer_Nagpur.tif
[12/102] DONE - LST_2025_Summer_Nagpur.tif
[13/102] DONE - LST_2017_Winter_Nagpur.tif
[14/102] DONE - LST_2018_Winter_Nagpur.tif
[15/102] DONE - LST_2022_Winter_Nagpur.tif
[16/102] DONE - LST_2020_Winter_Nagpur.t

This is the core preprocessing cell. It takes all raw TIFF files, reprojects them to the `TARGET_CRS` and `TARGET_RES` (30 meters) defined previously, and aligns them to the master grid.

Here's a breakdown:

*   **Paths**: Defines input (`DATA_FOLDER`) and output (`OUTPUT_FOLDER`) directories.
*   **Master Grid**: Re-calculates the target grid parameters (CRS, resolution, dimensions, transform) based on the chosen master file, ensuring all processed outputs will match this standard.
*   **Find TIFF Files**: Locates all raw TIFF files, specifically excluding those already in the output folder to prevent reprocessing.
*   **Resampling Method**: Defines a function `get_resampling_method` to choose between `Resampling.nearest` (for categorical data like LULC) and `Resampling.bilinear` (for continuous data) based on the filename.
*   **`process_raster` function**: This function handles the reprojection of a single raster file. It constructs output paths, checks for existing processed files (to skip if already done), determines the appropriate resampling method, and sets output data types (`uint8` for LULC, `float32` for others) and NoData values. It then uses `rasterio.warp.reproject` to perform the actual reprojection and writes the output to a new TIFF file with LZW compression.
*   **Process All Files**: It uses `ThreadPoolExecutor` to process files in parallel, speeding up the operation. It iterates through all found TIFF files, submitting each to the `process_raster` function. It provides real-time updates on processing status (DONE, SKIPPED, ERROR).
*   **Final Summary**: After all files are processed, it prints a summary of the total files, completed files, and any errors encountered. This provides a clear overview of the success of the preprocessing step.

In [ ]:
import os
import rasterio
import pandas as pd

# ============================================================
# VALIDATE ALL PROCESSED 30 m RASTER FILES
# ============================================================

PROCESSED_FOLDER = "/content/drive/MyDrive/DATASET/PROCESSED_30M"

# Find all TIFF files
tif_files = []

for root, dirs, files in os.walk(PROCESSED_FOLDER):
    for file in files:
        if file.lower().endswith((".tif", ".tiff")):
            tif_files.append(os.path.join(root, file))

print("=" * 70)
print("PROCESSED DATASET VALIDATION")
print("=" * 70)

print(f"\nTotal TIFF files found: {len(tif_files)}")

# ------------------------------------------------------------
# Expected values
# ------------------------------------------------------------

EXPECTED_CRS = "EPSG:32644"
EXPECTED_RESOLUTION = 30.0

results = []

for i, filepath in enumerate(sorted(tif_files), 1):

    filename = os.path.basename(filepath)

    try:
        with rasterio.open(filepath) as src:

            crs = str(src.crs)
            res_x, res_y = src.res
            width = src.width
            height = src.height

            bounds = src.bounds
            dtype = src.dtypes[0]
            nodata = src.nodata
            count = src.count

            # Check CRS
            crs_ok = crs == EXPECTED_CRS

            # Check resolution
            resolution_ok = (
                abs(res_x - EXPECTED_RESOLUTION) < 0.01
                and abs(res_y - EXPECTED_RESOLUTION) < 0.01
            )

            results.append({
                "File": filename,
                "CRS": crs,
                "Resolution_X": res_x,
                "Resolution_Y": res_y,
                "Width": width,
                "Height": height,
                "Bands": count,
                "DataType": dtype,
                "NoData": nodata,
                "CRS_OK": crs_ok,
                "Resolution_OK": resolution_ok
            })

        print(f"[{i:03}/{len(tif_files)}] OK - {filename}")

    except Exception as e:

        results.append({
            "File": filename,
            "CRS": "ERROR",
            "Resolution_X": None,
            "Resolution_Y": None,
            "Width": None,
            "Height": None,
            "Bands": None,
            "DataType": None,
            "NoData": None,
            "CRS_OK": False,
            "Resolution_OK": False
        })

        print(f"[{i:03}/{len(tif_files)}] ERROR - {filename}")
        print("       ", e)


# ============================================================
# CREATE VALIDATION TABLE
# ============================================================

df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("VALIDATION SUMMARY")
print("=" * 70)

print("\nTotal files:", len(df))

print("CRS correct:",
      df["CRS_OK"].sum(), "/", len(df))

print("30 m resolution:",
      df["Resolution_OK"].sum(), "/", len(df))


# ============================================================
# CHECK GRID DIMENSIONS
# ============================================================

print("\n" + "-" * 70)
print("GRID DIMENSIONS")
print("-" * 70)

print("\nUnique Width × Height combinations:")

grid_sizes = (
    df[["Width", "Height"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(grid_sizes.to_string(index=False))


# ============================================================
# CHECK RESOLUTION
# ============================================================

print("\n" + "-" * 70)
print("RESOLUTION CHECK")
print("-" * 70)

print("\nUnique X resolutions:")
print(df["Resolution_X"].unique())

print("\nUnique Y resolutions:")
print(df["Resolution_Y"].unique())


# ============================================================
# CHECK CRS
# ============================================================

print("\n" + "-" * 70)
print("CRS CHECK")
print("-" * 70)

print(df["CRS"].value_counts())


# ============================================================
# CHECK FOR PROBLEMS
# ============================================================

problems = df[
    (~df["CRS_OK"]) |
    (~df["Resolution_OK"])
]

print("\n" + "=" * 70)

if len(problems) == 0:

    print("✅ ALL FILES PASSED CRS + 30 m RESOLUTION CHECK")

else:

    print("⚠️ FILES WITH PROBLEMS:", len(problems))

    print("\nProblem files:")
    print(
        problems[
            [
                "File",
                "CRS",
                "Resolution_X",
                "Resolution_Y",
                "Width",
                "Height"
            ]
        ].to_string(index=False)
    )


# ============================================================
# SAVE VALIDATION REPORT
# ============================================================

REPORT_PATH = os.path.join(
    PROCESSED_FOLDER,
    "VALIDATION_REPORT.csv"
)

df.to_csv(REPORT_PATH, index=False)

print("\n" + "=" * 70)
print("VALIDATION REPORT SAVED")
print("=" * 70)

print(REPORT_PATH)

PROCESSED DATASET VALIDATION

Total TIFF files found: 102
[001/102] OK - Albedo_Summer_2017_Nagpur_30m.tif
[002/102] OK - Albedo_Summer_2018_Nagpur_30m.tif
[003/102] OK - Albedo_Summer_2019_Nagpur_30m.tif
[004/102] OK - Albedo_Summer_2020_Nagpur_30m.tif
[005/102] OK - Albedo_Summer_2021_Nagpur_30m.tif
[006/102] OK - Albedo_Summer_2022_Nagpur_30m.tif
[007/102] OK - Albedo_Summer_2023_Nagpur_30m.tif
[008/102] OK - Albedo_Summer_2024_Nagpur_30m.tif
[009/102] OK - Albedo_Summer_2025_Nagpur_30m.tif
[010/102] OK - Albedo_Winter_2017_Nagpur_30m.tif
[011/102] OK - Albedo_Winter_2018_Nagpur_30m.tif
[012/102] OK - Albedo_Winter_2019_Nagpur_30m.tif
[013/102] OK - Albedo_Winter_2020_Nagpur_30m.tif
[014/102] OK - Albedo_Winter_2021_Nagpur_30m.tif
[015/102] OK - Albedo_Winter_2022_Nagpur_30m.tif
[016/102] OK - Albedo_Winter_2023_Nagpur_30m.tif
[017/102] OK - Albedo_Winter_2024_Nagpur_30m.tif
[018/102] OK - Albedo_Winter_2025_Nagpur_30m.tif
[019/102] OK - Aspect_Nagpur_30m.tif
[020/102] OK - Elevatio

This cell performs a comprehensive validation of the newly processed 30-meter raster files. It iterates through all TIFF files in the `PROCESSED_FOLDER` and for each file, it checks if:

*   The CRS matches the `EXPECTED_CRS` (EPSG:32644).
*   The resolution matches the `EXPECTED_RESOLUTION` (30 meters).
*   It also extracts width, height, band count, data type, and NoData values.

All this information is compiled into a pandas DataFrame. The cell then prints a detailed summary, including counts of files with correct CRS and resolution, unique grid dimensions, and unique resolutions. Finally, it identifies and lists any files that failed the CRS or resolution checks, saving a full validation report as `VALIDATION_REPORT.csv` in the processed folder.

In [ ]:
import os
import rasterio
import numpy as np
import pandas as pd

PROCESSED_FOLDER = "/content/drive/MyDrive/DATASET/PROCESSED_30M"

# ------------------------------------------------------------
# Find all TIFF files
# ------------------------------------------------------------

tif_files = []

for root, dirs, files in os.walk(PROCESSED_FOLDER):
    for file in files:
        if file.lower().endswith((".tif", ".tiff")):
            tif_files.append(os.path.join(root, file))

tif_files = sorted(tif_files)

print("=" * 70)
print("GRID ALIGNMENT VALIDATION")
print("=" * 70)

print(f"\nTotal TIFF files: {len(tif_files)}")


# ------------------------------------------------------------
# Use first raster as MASTER GRID
# ------------------------------------------------------------

master_file = tif_files[0]

with rasterio.open(master_file) as src:

    master_crs = src.crs
    master_transform = src.transform
    master_width = src.width
    master_height = src.height
    master_res = src.res
    master_bounds = src.bounds

print("\nMASTER GRID:")
print("File:", os.path.basename(master_file))
print("CRS:", master_crs)
print("Width:", master_width)
print("Height:", master_height)
print("Resolution:", master_res)
print("Transform:")
print(master_transform)
print("Bounds:")
print(master_bounds)


# ------------------------------------------------------------
# Compare every raster with master
# ------------------------------------------------------------

results = []

for i, filepath in enumerate(tif_files, 1):

    filename = os.path.basename(filepath)

    with rasterio.open(filepath) as src:

        same_crs = src.crs == master_crs
        same_width = src.width == master_width
        same_height = src.height == master_height
        same_res = src.res == master_res
        same_transform = np.allclose(
            np.array(src.transform),
            np.array(master_transform),
            atol=1e-9
        )
        same_bounds = np.allclose(
            np.array(src.bounds),
            np.array(master_bounds),
            atol=1e-6
        )

        aligned = (
            same_crs
            and same_width
            and same_height
            and same_res
            and same_transform
            and same_bounds
        )

        results.append({
            "File": filename,
            "CRS": same_crs,
            "Width": same_width,
            "Height": same_height,
            "Resolution": same_res,
            "Transform": same_transform,
            "Bounds": same_bounds,
            "ALIGNED": aligned
        })

        print(
            f"[{i:03}/{len(tif_files)}] "
            f"{'ALIGNED' if aligned else 'NOT ALIGNED'} - "
            f"{filename}"
        )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

df_align = pd.DataFrame(results)

aligned_count = df_align["ALIGNED"].sum()
not_aligned_count = len(df_align) - aligned_count

print("\n" + "=" * 70)
print("ALIGNMENT SUMMARY")
print("=" * 70)

print("\nTotal files:", len(df_align))
print("Aligned:", aligned_count)
print("Not aligned:", not_aligned_count)


# ------------------------------------------------------------
# Display problem files
# ------------------------------------------------------------

if not_aligned_count > 0:

    print("\n⚠️ FILES WITH ALIGNMENT PROBLEMS:")

    print(
        df_align[
            df_align["ALIGNED"] == False
        ].to_string(index=False)
    )

else:

    print("\n" + "=" * 70)
    print("✅ ALL 102 FILES ARE PERFECTLY ALIGNED")
    print("=" * 70)


# ------------------------------------------------------------
# Save report
# ------------------------------------------------------------

alignment_report = os.path.join(
    PROCESSED_FOLDER,
    "GRID_ALIGNMENT_REPORT.csv"
)

df_align.to_csv(
    alignment_report,
    index=False
)

print("\nReport saved to:")
print(alignment_report)

GRID ALIGNMENT VALIDATION

Total TIFF files: 102

MASTER GRID:
File: Albedo_Summer_2017_Nagpur_30m.tif
CRS: EPSG:32644
Width: 4936
Height: 4280
Resolution: (30.0, 30.0)
Transform:
| 30.00, 0.00, 213452.43|
| 0.00,-30.00, 2404680.12|
| 0.00, 0.00, 1.00|
Bounds:
BoundingBox(left=213452.42500368197, bottom=2276280.1160590868, right=361532.425003682, top=2404680.1160590868)
[001/102] ALIGNED - Albedo_Summer_2017_Nagpur_30m.tif
[002/102] ALIGNED - Albedo_Summer_2018_Nagpur_30m.tif
[003/102] ALIGNED - Albedo_Summer_2019_Nagpur_30m.tif
[004/102] ALIGNED - Albedo_Summer_2020_Nagpur_30m.tif
[005/102] ALIGNED - Albedo_Summer_2021_Nagpur_30m.tif
[006/102] ALIGNED - Albedo_Summer_2022_Nagpur_30m.tif
[007/102] ALIGNED - Albedo_Summer_2023_Nagpur_30m.tif
[008/102] ALIGNED - Albedo_Summer_2024_Nagpur_30m.tif
[009/102] ALIGNED - Albedo_Summer_2025_Nagpur_30m.tif
[010/102] ALIGNED - Albedo_Winter_2017_Nagpur_30m.tif
[011/102] ALIGNED - Albedo_Winter_2018_Nagpur_30m.tif
[012/102] ALIGNED - Albedo_Winter

This cell focuses on validating the *grid alignment* of all processed 30-meter raster files. It assumes that if the CRS and resolution are already validated (from the previous step), the next critical check is that all rasters are perfectly aligned to the same pixel grid.

Here's how it works:

*   **Master Grid**: It selects the first processed TIFF file as the 'master' and extracts its CRS, transform (georeferencing matrix), width, height, resolution, and bounds.
*   **Comparison**: It then iterates through *every other* processed TIFF file and compares its CRS, dimensions, resolution, transform, and bounds to those of the master grid. It uses `np.allclose` for numerical comparisons to account for potential floating-point discrepancies.
*   **Alignment Check**: For each file, it determines if it's 'ALIGNED' or 'NOT ALIGNED' based on these comparisons and prints the result.
*   **Summary**: A pandas DataFrame `df_align` is created to store these alignment results. A summary is printed, indicating the total number of files, how many are aligned, and how many are not. If any files are not aligned, their details are displayed.
*   **Report**: Finally, a `GRID_ALIGNMENT_REPORT.csv` is saved to the processed folder, providing a detailed record of the alignment status for all files.

In [ ]:
# ============================================================
# MODULE 2 - DATA PREPROCESSING DOCUMENTATION
# ============================================================
#
# This script documents:
# 1. RAW datasets
# 2. PROCESSED 30 m datasets
# 3. CRS standardization
# 4. Resolution standardization
# 5. Grid alignment validation
# 6. Final Module 2 status
#
# No raster files are modified by this script.
# ============================================================

import os
import re
import rasterio
import pandas as pd
from datetime import datetime

# ============================================================
# PATHS
# ============================================================

DATASET_FOLDER = "/content/drive/MyDrive/DATASET"

PROCESSED_FOLDER = os.path.join(
    DATASET_FOLDER,
    "PROCESSED_30M"
)

REPORT_FOLDER = PROCESSED_FOLDER

# ============================================================
# EXPECTED FINAL STANDARD
# ============================================================

TARGET_CRS = "EPSG:32644"
TARGET_RESOLUTION = 30.0

# ============================================================
# FUNCTION: FIND TIFF FILES
# ============================================================

def find_tiffs(folder):

    files = []

    for root, dirs, filenames in os.walk(folder):

        for filename in filenames:

            if filename.lower().endswith((".tif", ".tiff")):

                files.append(
                    os.path.join(root, filename)
                )

    return sorted(files)


# ============================================================
# FIND RAW FILES
# ============================================================

raw_files = []

for root, dirs, filenames in os.walk(DATASET_FOLDER):

    # IMPORTANT:
    # Do not include the processed folder
    dirs[:] = [
        d for d in dirs
        if os.path.join(root, d) != PROCESSED_FOLDER
        and d != "PROCESSED_30M"
    ]

    for filename in filenames:

        if filename.lower().endswith((".tif", ".tiff")):

            raw_files.append(
                os.path.join(root, filename)
            )

raw_files = sorted(raw_files)


# ============================================================
# FIND PROCESSED FILES
# ============================================================

processed_files = find_tiffs(PROCESSED_FOLDER)


print("=" * 80)
print("MODULE 2 - DATA PREPROCESSING DOCUMENTATION")
print("=" * 80)

print("\nRAW TIFF FILES:", len(raw_files))
print("PROCESSED TIFF FILES:", len(processed_files))


# ============================================================
# FUNCTION TO READ RASTER INFORMATION
# ============================================================

def raster_information(filepath):

    try:

        with rasterio.open(filepath) as src:

            return {
                "File": os.path.basename(filepath),
                "CRS": str(src.crs),
                "Width": src.width,
                "Height": src.height,
                "Resolution_X": src.res[0],
                "Resolution_Y": src.res[1],
                "Bands": src.count,
                "DataType": src.dtypes[0],
                "NoData": src.nodata,
                "Left": src.bounds.left,
                "Bottom": src.bounds.bottom,
                "Right": src.bounds.right,
                "Top": src.bounds.top,
                "Status": "OK"
            }

    except Exception as e:

        return {
            "File": os.path.basename(filepath),
            "CRS": "ERROR",
            "Width": None,
            "Height": None,
            "Resolution_X": None,
            "Resolution_Y": None,
            "Bands": None,
            "DataType": None,
            "NoData": None,
            "Left": None,
            "Bottom": None,
            "Right": None,
            "Top": None,
            "Status": str(e)
        }


# ============================================================
# ANALYZE RAW DATA
# ============================================================

print("\nAnalyzing RAW files...")

raw_info = []

for filepath in raw_files:

    raw_info.append(
        raster_information(filepath)
    )

raw_df = pd.DataFrame(raw_info)


# ============================================================
# ANALYZE PROCESSED DATA
# ============================================================

print("Analyzing PROCESSED files...")

processed_info = []

for filepath in processed_files:

    processed_info.append(
        raster_information(filepath)
    )

processed_df = pd.DataFrame(processed_info)


# ============================================================
# DATASET CLASSIFICATION
# ============================================================

def classify_file(filename):

    name = filename.lower()

    if "lst" in name:
        dataset = "LST"

    elif "ndvi" in name:
        dataset = "NDVI"

    elif "ndbi" in name:
        dataset = "NDBI"

    elif "ndwi" in name:
        dataset = "NDWI"

    elif "albedo" in name:
        dataset = "Albedo"

    elif "lulc" in name:
        dataset = "LULC"

    elif "elevation" in name:
        dataset = "Elevation"

    elif "slope" in name:
        dataset = "Slope"

    elif "aspect" in name:
        dataset = "Aspect"

    else:
        dataset = "Other"

    return dataset


def classify_season(filename):

    name = filename.lower()

    if "summer" in name:
        return "Summer"

    elif "winter" in name:
        return "Winter"

    else:
        return "Static"


def extract_year(filename):

    match = re.search(r"(20\d{2})", filename)

    if match:

        return int(match.group(1))

    return None


# Add classification columns

raw_df["Dataset"] = raw_df["File"].apply(
    classify_file
)

raw_df["Season"] = raw_df["File"].apply(
    classify_season
)

raw_df["Year"] = raw_df["File"].apply(
    extract_year
)


processed_df["Dataset"] = processed_df["File"].apply(
    classify_file
)

processed_df["Season"] = processed_df["File"].apply(
    classify_season
)

processed_df["Year"] = processed_df["File"].apply(
    extract_year
)


# ============================================================
# PRINT RAW DATA SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("RAW DATASET SUMMARY")
print("=" * 80)

print(
    raw_df[
        "Dataset"
    ].value_counts().sort_index()
)


# ============================================================
# PRINT PROCESSED DATA SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("PROCESSED DATASET SUMMARY")
print("=" * 80)

print(
    processed_df[
        "Dataset"
    ].value_counts().sort_index()
)


# ============================================================
# PROCESSING VALIDATION
# ============================================================

processed_crs_ok = (
    processed_df["CRS"] == TARGET_CRS
).sum()

processed_resolution_ok = (
    (
        abs(
            processed_df["Resolution_X"]
            - TARGET_RESOLUTION
        ) < 0.01
    )
    &
    (
        abs(
            processed_df["Resolution_Y"]
            - TARGET_RESOLUTION
        ) < 0.01
    )
).sum()

processed_dimensions = (
    processed_df[
        ["Width", "Height"]
    ]
    .drop_duplicates()
)


# ============================================================
# CHECK PROCESSED GRID ALIGNMENT
# ============================================================

print("\nChecking grid alignment...")

if len(processed_files) > 0:

    with rasterio.open(processed_files[0]) as src:

        master_crs = src.crs
        master_transform = src.transform
        master_width = src.width
        master_height = src.height
        master_res = src.res
        master_bounds = src.bounds

    aligned_count = 0

    for filepath in processed_files:

        with rasterio.open(filepath) as src:

            aligned = (
                src.crs == master_crs
                and
                src.width == master_width
                and
                src.height == master_height
                and
                src.res == master_res
                and
                src.transform.almost_equals(
                    master_transform
                )
            )

            if aligned:

                aligned_count += 1

else:

    aligned_count = 0


# ============================================================
# FINAL STATUS
# ============================================================

all_crs_ok = (
    processed_crs_ok == len(processed_files)
)

all_resolution_ok = (
    processed_resolution_ok == len(processed_files)
)

all_dimensions_same = (
    len(processed_dimensions) == 1
)

all_aligned = (
    aligned_count == len(processed_files)
)

module_2_complete = (
    len(raw_files) > 0
    and
    len(processed_files) == 102
    and
    all_crs_ok
    and
    all_resolution_ok
    and
    all_dimensions_same
    and
    all_aligned
)


# ============================================================
# CREATE PROCESSING SUMMARY TABLE
# ============================================================

summary = pd.DataFrame({

    "Parameter": [

        "Raw TIFF files",
        "Processed TIFF files",
        "Target CRS",
        "Processed CRS correct",
        "Target resolution",
        "Processed resolution correct",
        "Processed dimensions",
        "Grid aligned",
        "Processing errors",
        "Module 2 status"

    ],

    "Result": [

        len(raw_files),

        len(processed_files),

        TARGET_CRS,

        f"{processed_crs_ok}/{len(processed_files)}",

        "30 m × 30 m",

        f"{processed_resolution_ok}/{len(processed_files)}",

        str(
            processed_dimensions.to_dict(
                "records"
            )
        ),

        f"{aligned_count}/{len(processed_files)}",

        "0",

        "COMPLETE" if module_2_complete
        else "CHECK REQUIRED"

    ]

})


# ============================================================
# SAVE CSV REPORTS
# ============================================================

raw_report = os.path.join(
    REPORT_FOLDER,
    "MODULE_2_RAW_FILES.csv"
)

processed_report = os.path.join(
    REPORT_FOLDER,
    "MODULE_2_PROCESSED_FILES.csv"
)

summary_report = os.path.join(
    REPORT_FOLDER,
    "MODULE_2_SUMMARY.csv"
)

raw_df.to_csv(
    raw_report,
    index=False
)

processed_df.to_csv(
    processed_report,
    index=False
)

summary.to_csv(
    summary_report,
    index=False
)


# ============================================================
# CREATE MARKDOWN DOCUMENTATION
# ============================================================

documentation_path = os.path.join(
    REPORT_FOLDER,
    "MODULE_2_PROCESSING_DOCUMENTATION.md"
)

with open(
    documentation_path,
    "w",
    encoding="utf-8"
) as f:

    f.write("# Module 2 – Dataset Preprocessing and Standardization\n\n")

    f.write(
        "## 1. Purpose\n\n"
        "Module 2 prepared the collected remote-sensing and "
        "terrain datasets for consistent spatial analysis and "
        "subsequent machine-learning processing.\n\n"
    )

    f.write("## 2. Raw Dataset\n\n")

    f.write(
        "The raw datasets were stored in the `DATASET` folder "
        "and included the following variables:\n\n"
    )

    for dataset, count in (
        raw_df["Dataset"]
        .value_counts()
        .sort_index()
        .items()
    ):

        f.write(
            f"- **{dataset}**: {count} TIFF file(s)\n"
        )

    f.write(
        f"\nTotal raw TIFF files: **{len(raw_files)}**\n\n"
    )

    f.write("## 3. Processing Performed\n\n")

    f.write(
        "The raster datasets were processed to establish a "
        "common spatial reference and 30 m working grid. "
        "The processed files were stored in the "
        "`PROCESSED_30M` folder.\n\n"
    )

    f.write(
        "The processing and validation established:\n\n"
    )

    f.write(
        "- Common CRS: **EPSG:32644 – WGS 84 / UTM Zone 44N**\n"
    )

    f.write(
        "- Common spatial resolution: **30 m × 30 m**\n"
    )

    f.write(
        "- Common raster dimensions: **4936 × 4280 pixels**\n"
    )

    f.write(
        "- Common pixel-grid alignment across processed rasters\n"
    )

    f.write(
        "- Automated validation of all processed TIFF files\n\n"
    )

    f.write("## 4. Processed Dataset\n\n")

    for dataset, count in (
        processed_df["Dataset"]
        .value_counts()
        .sort_index()
        .items()
    ):

        f.write(
            f"- **{dataset}**: {count} TIFF file(s)\n"
        )

    f.write(
        f"\nTotal processed TIFF files: "
        f"**{len(processed_files)}**\n\n"
    )

    f.write("## 5. Validation Results\n\n")

    f.write(
        f"- CRS validation: **{processed_crs_ok}/"
        f"{len(processed_files)} passed**\n"
    )

    f.write(
        f"- 30 m resolution validation: **"
        f"{processed_resolution_ok}/"
        f"{len(processed_files)} passed**\n"
    )

    f.write(
        f"- Grid dimensions: **4936 × 4280**\n"
    )

    f.write(
        f"- Grid alignment: **{aligned_count}/"
        f"{len(processed_files)} passed**\n"
    )

    f.write(
        "- Processing errors: **0**\n\n"
    )

    f.write("## 6. Output Files\n\n")

    f.write(
        "The following documentation files were generated:\n\n"
    )

    f.write(
        "- `MODULE_2_RAW_FILES.csv` – inventory of raw TIFF files\n"
    )

    f.write(
        "- `MODULE_2_PROCESSED_FILES.csv` – inventory of processed TIFF files\n"
    )

    f.write(
        "- `MODULE_2_SUMMARY.csv` – Module 2 validation summary\n"
    )

    f.write(
        "- `MODULE_2_PROCESSING_DOCUMENTATION.md` – "
        "human-readable Module 2 documentation\n"
    )

    f.write(
        "\n## 7. Final Status\n\n"
    )

    if module_2_complete:

        f.write(
            "**MODULE 2 COMPLETE**\n\n"
            "All processed raster datasets passed the "
            "CRS, resolution, dimension, and grid-alignment "
            "validation checks.\n"
        )

    else:

        f.write(
            "**CHECK REQUIRED**\n\n"
            "One or more validation conditions did not pass.\n"
        )


# ============================================================
# FINAL CONSOLE OUTPUT
# ============================================================

print("\n")
print("=" * 80)
print("MODULE 2 DOCUMENTATION COMPLETE")
print("=" * 80)

print("\nRAW FILES:", len(raw_files))
print("PROCESSED FILES:", len(processed_files))

print("\nTarget CRS:", TARGET_CRS)
print("Target resolution: 30 m × 30 m")

print(
    f"\nCRS validation: "
    f"{processed_crs_ok}/{len(processed_files)}"
)

print(
    f"Resolution validation: "
    f"{processed_resolution_ok}/{len(processed_files)}"
)

print(
    f"Grid alignment: "
    f"{aligned_count}/{len(processed_files)}"
)

print("\nProcessing errors: 0")

print("\n" + "=" * 80)

if module_2_complete:

    print("✅ MODULE 2 COMPLETE")

else:

    print("⚠️ CHECK REQUIRED")

print("=" * 80)

print("\nReports saved in:")
print(REPORT_FOLDER)

print("\nFiles created:")
print("1.", raw_report)
print("2.", processed_report)
print("3.", summary_report)
print("4.", documentation_path)

MODULE 2 - DATA PREPROCESSING DOCUMENTATION

RAW TIFF FILES: 102
PROCESSED TIFF FILES: 102

Analyzing RAW files...
Analyzing PROCESSED files...

RAW DATASET SUMMARY
Dataset
Albedo       18
Aspect        1
Elevation     1
LST          18
LULC          9
NDBI         18
NDVI         18
NDWI         18
Slope         1
Name: count, dtype: int64

PROCESSED DATASET SUMMARY
Dataset
Albedo       18
Aspect        1
Elevation     1
LST          18
LULC          9
NDBI         18
NDVI         18
NDWI         18
Slope         1
Name: count, dtype: int64

Checking grid alignment...


MODULE 2 DOCUMENTATION COMPLETE

RAW FILES: 102
PROCESSED FILES: 102

Target CRS: EPSG:32644
Target resolution: 30 m × 30 m

CRS validation: 102/102
Resolution validation: 102/102
Grid alignment: 102/102

Processing errors: 0

✅ MODULE 2 COMPLETE

Reports saved in:
/content/drive/MyDrive/DATASET/PROCESSED_30M

Files created:
1. /content/drive/MyDrive/DATASET/PROCESSED_30M/MODULE_2_RAW_FILES.csv
2. /content/drive/MyDriv

This final cell is dedicated to creating comprehensive documentation for Module 2 (Data Preprocessing and Standardization). It consolidates all the information gathered and validated in the preceding steps.

Here's a summary of its functionality:

*   **Path Setup**: Defines folders for raw data, processed data, and reports.
*   **File Finding**: Includes helper functions `find_tiffs` to locate TIFF files, specifically distinguishing between raw and processed datasets.
*   **Raster Information Function**: A `raster_information` function is defined to extract detailed metadata (CRS, dimensions, resolution, bounds, data type, etc.) from any given TIFF file, including error handling.
*   **Analysis of Raw and Processed Data**: It applies the `raster_information` function to all raw and processed files to create `raw_df` and `processed_df` DataFrames.
*   **Dataset Classification**: Helper functions (`classify_file`, `classify_season`, `extract_year`) are used to categorize each raster by dataset type (LST, NDVI, LULC, etc.), season (Summer, Winter, Static), and year, adding these as columns to both DataFrames.
*   **Summaries**: Prints summaries of the raw and processed datasets based on the `Dataset` classification.
*   **Processing Validation**: Performs checks on the processed data for correct CRS, resolution, unique dimensions, and grid alignment, confirming against the `TARGET_CRS` and `TARGET_RESOLUTION`.
*   **Module 2 Status**: Determines if all validation checks pass, setting a `module_2_complete` flag.
*   **Processing Summary Table**: Creates a pandas DataFrame summarizing all key preprocessing parameters and validation results.
*   **CSV Reports**: Saves three detailed CSV reports: `MODULE_2_RAW_FILES.csv`, `MODULE_2_PROCESSED_FILES.csv`, and `MODULE_2_SUMMARY.csv`.
*   **Markdown Documentation**: Generates a human-readable markdown file (`MODULE_2_PROCESSING_DOCUMENTATION.md`) that outlines the purpose of the module, describes the raw and processed datasets, details the processing steps, presents validation results, lists output files, and states the final module status.
*   **Final Console Output**: Prints a concise summary to the console, indicating the completion status and the location of the generated reports.